In [ ]:
%pip install --upgrade azure-ai-ml
%pip install setuptools==81.0.0
%pip install azureml-fsspec==1.3.1

[notice] A new release of pip is available: 26.1.2 -> 26.2.1
[notice] To update, run: /anaconda/envs/azureml_py310_sdkv2/bin/python -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [1]:
from azure.ai.ml.entities import AzureBlobDatastore
from azure.ai.ml import command, Input, Output, MLClient
from azure.identity import DefaultAzureCredential
import pandas as pd
import os 
import logging
import mlflow
import time
from azure.ai.ml.entities import Data
from azure.ai.ml.constants import AssetTypes, InputOutputModes
import mltable
from mltable import MLTableHeaders, MLTableFileEncoding, DataType

ml_client = MLClient.from_config(credential=DefaultAzureCredential())

/anaconda/envs/azureml_py310_sdkv2/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/anaconda/envs/azureml_py310_sdkv2/lib/python3.10/site-packages/azureml/dataprep/api/_loggerfactory.py:8: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources
Found the config file in: /config.json


In [2]:
logging.getLogger("azure").setLevel(logging.DEBUG)

In [3]:
mlflow_tracking_uri = ml_client.workspaces.get(ml_client.workspace_name).mlflow_tracking_uri

In [4]:
mlflow.set_tracking_uri(mlflow_tracking_uri)

In [5]:
experiment_name = 'hello-world-example'
mlflow.set_experiment(experiment_name)

<Experiment: artifact_location='', creation_time=1789034776680, effective_trace_archival_retention=None, experiment_id='004b62c3-727a-4720-8706-671e4f349eac', last_update_time=None, lifecycle_stage='active', name='hello-world-example', tags={}, trace_location=None, workspace='default'>

In [6]:
# Start the run
mlflow_run = mlflow.start_run()
# Log metrics or other information
mlflow.log_metric('mymetric', 1)
# End run 
mlflow.end_run()

🏃 View run clever_hominy_gbxsw61h at: https://centralindia.api.azureml.ms/mlflow/v2.0/subscriptions/5cc08b97-8906-4620-861c-088452556a7b/resourceGroups/demogroup/providers/Microsoft.MachineLearningServices/workspaces/mlw-mlops-3nftht/#/experiments/004b62c3-727a-4720-8706-671e4f349eac/runs/9d53daf1-a6de-4823-bcd1-8b3956c8c0c4
🧪 View experiment at: https://centralindia.api.azureml.ms/mlflow/v2.0/subscriptions/5cc08b97-8906-4620-861c-088452556a7b/resourceGroups/demogroup/providers/Microsoft.MachineLearningServices/workspaces/mlw-mlops-3nftht/#/experiments/004b62c3-727a-4720-8706-671e4f349eac


In [7]:
#Use context manager paradigm
import mlflow
mlflow.set_experiment("mlflow-experiment")

# Start the run, log metrics, end the run
with mlflow.start_run() as run:
    # Run started when context manager is entered, and ended when context manager exits
    mlflow.log_metric('mymetric', 1)
    mlflow.log_metric('anothermetric',1)
    pass

2026/09/10 10:52:53 INFO mlflow.tracking.fluent: Experiment with name 'mlflow-experiment' does not exist. Creating a new experiment.


🏃 View run polite_lamp_yfsm9755 at: https://centralindia.api.azureml.ms/mlflow/v2.0/subscriptions/5cc08b97-8906-4620-861c-088452556a7b/resourceGroups/demogroup/providers/Microsoft.MachineLearningServices/workspaces/mlw-mlops-3nftht/#/experiments/a8f0136e-adf3-476e-ab9a-622b722464c8/runs/f07ee5a6-f0f3-4ef2-bc7c-8ee6acf1f549
🧪 View experiment at: https://centralindia.api.azureml.ms/mlflow/v2.0/subscriptions/5cc08b97-8906-4620-861c-088452556a7b/resourceGroups/demogroup/providers/Microsoft.MachineLearningServices/workspaces/mlw-mlops-3nftht/#/experiments/a8f0136e-adf3-476e-ab9a-622b722464c8


In [8]:
#You can specify the run name of your choice as well
with mlflow.start_run(run_name="iris-classifier-random-forest") as run:
    mlflow.log_metric('mymetric', 1)
    mlflow.log_metric('anothermetric',1)

🏃 View run iris-classifier-random-forest at: https://centralindia.api.azureml.ms/mlflow/v2.0/subscriptions/5cc08b97-8906-4620-861c-088452556a7b/resourceGroups/demogroup/providers/Microsoft.MachineLearningServices/workspaces/mlw-mlops-3nftht/#/experiments/a8f0136e-adf3-476e-ab9a-622b722464c8/runs/ce2814e4-0f61-4872-aef0-bf79cb123e0c
🧪 View experiment at: https://centralindia.api.azureml.ms/mlflow/v2.0/subscriptions/5cc08b97-8906-4620-861c-088452556a7b/resourceGroups/demogroup/providers/Microsoft.MachineLearningServices/workspaces/mlw-mlops-3nftht/#/experiments/a8f0136e-adf3-476e-ab9a-622b722464c8


In [9]:
#Loggin parameters
params = {
    "num_epochs": 20,
    "dropout_rate": .6,
    "objective": "binary_crossentropy"
    }
with mlflow.start_run(run_name = "parameter-loggin-random-forest") as run:
    mlflow.log_params(params)

🏃 View run parameter-loggin-random-forest at: https://centralindia.api.azureml.ms/mlflow/v2.0/subscriptions/5cc08b97-8906-4620-861c-088452556a7b/resourceGroups/demogroup/providers/Microsoft.MachineLearningServices/workspaces/mlw-mlops-3nftht/#/experiments/a8f0136e-adf3-476e-ab9a-622b722464c8/runs/dd002be2-7964-434c-bbca-1248f28a57ed
🧪 View experiment at: https://centralindia.api.azureml.ms/mlflow/v2.0/subscriptions/5cc08b97-8906-4620-861c-088452556a7b/resourceGroups/demogroup/providers/Microsoft.MachineLearningServices/workspaces/mlw-mlops-3nftht/#/experiments/a8f0136e-adf3-476e-ab9a-622b722464c8


In [13]:
#When you want to log metrics from multiple nodes, or few nodes trying to log a lot of metrics
# at once, async_logging can be enabled, this allows you to log metrics without waiting for
# the metrics to materialize in bacend.
# This flag should be used in scripts.
mlflow.config.enable_async_logging()

with mlflow.start_run(run_name = "loggin-metric-asynchronously-globally") as run:
    # (...)
    # You can use all fluent syntax or MlflowClient APIs and all of them will log metrics in asynchronous fashion.
    mlflow.log_metric("metric1", 9.42)

🏃 View run loggin-metric-asynchronously-globally at: https://centralindia.api.azureml.ms/mlflow/v2.0/subscriptions/5cc08b97-8906-4620-861c-088452556a7b/resourceGroups/demogroup/providers/Microsoft.MachineLearningServices/workspaces/mlw-mlops-3nftht/#/experiments/a8f0136e-adf3-476e-ab9a-622b722464c8/runs/544a0f28-dd17-43ca-b218-2f44d0c704fb
🧪 View experiment at: https://centralindia.api.azureml.ms/mlflow/v2.0/subscriptions/5cc08b97-8906-4620-861c-088452556a7b/resourceGroups/demogroup/providers/Microsoft.MachineLearningServices/workspaces/mlw-mlops-3nftht/#/experiments/a8f0136e-adf3-476e-ab9a-622b722464c8


In [14]:
#You can also log a specific metric asynchornously by setting the flag synchronous to false

with mlflow.start_run(run_name = "logging-sepcific-metric-asynchronously") as run:
    # (...)
    # You can use all fluent syntax or MlflowClient APIs and all of them will log metrics in asynchronous fashion.
    mlflow.log_metric("metric1", 9.42,synchronous=False)

🏃 View run logging-sepcific-metric-asynchronously at: https://centralindia.api.azureml.ms/mlflow/v2.0/subscriptions/5cc08b97-8906-4620-861c-088452556a7b/resourceGroups/demogroup/providers/Microsoft.MachineLearningServices/workspaces/mlw-mlops-3nftht/#/experiments/a8f0136e-adf3-476e-ab9a-622b722464c8/runs/08730183-4b92-40e0-b40c-8c8b640ca417
🧪 View experiment at: https://centralindia.api.azureml.ms/mlflow/v2.0/subscriptions/5cc08b97-8906-4620-861c-088452556a7b/resourceGroups/demogroup/providers/Microsoft.MachineLearningServices/workspaces/mlw-mlops-3nftht/#/experiments/a8f0136e-adf3-476e-ab9a-622b722464c8


In [16]:
# You can asynchornously log one metric or log a batch of metric asynchronoulsly as well

from mlflow.entities import Metric

with mlflow.start_run(run_name = "logging-metrics-batch-asynchronously") as current_run:
    mlflow_client = mlflow.tracking.MlflowClient()

    metrics = {"metric-0": 3.14, "metric-1": 6.28}
    timestamp = int(time.time() * 1000)
    metrics_arr = [Metric(key, value, timestamp, 0) for key, value in metrics.items()]

    run_operation = mlflow_client.log_batch(
        run_id=current_run.info.run_id,
        metrics=metrics_arr,
        #Optional when global async logging flag is set using - mlflow.enable_async_logging()
        synchronous=False,
    )

🏃 View run logging-metrics-batch-asynchronously at: https://centralindia.api.azureml.ms/mlflow/v2.0/subscriptions/5cc08b97-8906-4620-861c-088452556a7b/resourceGroups/demogroup/providers/Microsoft.MachineLearningServices/workspaces/mlw-mlops-3nftht/#/experiments/a8f0136e-adf3-476e-ab9a-622b722464c8/runs/4e84c1ac-64e6-471b-92d0-ad489a6c9493
🧪 View experiment at: https://centralindia.api.azureml.ms/mlflow/v2.0/subscriptions/5cc08b97-8906-4620-861c-088452556a7b/resourceGroups/demogroup/providers/Microsoft.MachineLearningServices/workspaces/mlw-mlops-3nftht/#/experiments/a8f0136e-adf3-476e-ab9a-622b722464c8


In [18]:
import os    
os.getcwd()

'/mnt/batch/tasks/shared/LS_root/mounts/clusters/mlops-ci001/code/Users/lishadalve/credit_risk_detection/MLflow'

In [22]:
#In general files in mlflow are called artifacts
# You can log artifacts in multiple ways
dictionary = {"Name":"Harshad","Age":29,"Address":"69 Shyam Nagar Main, Sukhliya"}
with mlflow.start_run(run_name = "logging-artifacts-from-mlflow") as current_run:
    mlflow.log_artifact("/mnt/batch/tasks/shared/LS_root/mounts/clusters/mlops-ci001/code/Users/lishadalve/credit_risk_detection/README.md")
    mlflow.log_artifacts("./data")
    mlflow.log_dict(dictionary, "bio.yaml")
    mlflow.log_text("This is to certify that Mr. Harshad Paymode has successfully cleared MLOps Associate Engineer (AI-300) Exam","certificate.txt")

In [25]:
# MLflow introduces the concept of models as a way to package all the artifacts 
# required for a given model to function. Models in MLflow are always a folder with an 
# arbitrary number of files, depending on the framework used to generate the model. 
# Logging models has the advantage of tracking all the elements of the model as a single 
# entity that can be registered and then deployed.

# os.environ["AZUREML_ARTIFACTS_DEFAULT_TIMEOUT"] = 300 (sec) by default
# If mlflow is unable to log the complete model (all artifiacts) withing above time limit, it
# will throw time out error which can be adjusted by setting up the above env variable


In [26]:
client = mlflow.tracking.MlflowClient()
client.list_artifacts("0f9ff8ca-b20a-4a4b-9707-cc1ccbf8c774")

[<FileInfo: file_size=-1, is_dir=False, path='.amlignore'>,
 <FileInfo: file_size=-1, is_dir=False, path='.amlignore.amltmp'>,
 <FileInfo: file_size=-1, is_dir=False, path='README.md'>,
 <FileInfo: file_size=-1, is_dir=False, path='bio.yaml'>,
 <FileInfo: file_size=-1, is_dir=False, path='certificate.txt'>,
 <FileInfo: file_size=-1, is_dir=False, path='ratings.csv'>]

In [28]:
run = mlflow.get_run("4e84c1ac-64e6-471b-92d0-ad489a6c9493")
run.data.metrics

{'metric-1': 6.28, 'metric-0': 3.14}

In [29]:
file_path = mlflow.artifacts.download_artifacts(run_id="0f9ff8ca-b20a-4a4b-9707-cc1ccbf8c774", artifact_path="certificate.txt")